<a href="https://colab.research.google.com/github/AgathaCRuiz/Voice-Powered-Labs/blob/main/03_meeting_minutes_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Assistente de Reuniões Inteligente
**Pipeline:** Gravação → Whisper (STT) → Groq (análise) → gTTS (confirmação por voz) → Ata em PDF

---

## 0. Instalação das dependências

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q
!pip install gTTS -q
!pip install fpdf2 -q
!pip install -q -U google-genai
!apt-get install -y ffmpeg -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.9/750.9 kB 15.8 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.

## 1. Configuração

In [ ]:
from google.colab import userdata

# --- Configurações gerais ---
IDIOMA_GRAVACAO   = 'pt'        # idioma para o Whisper
IDIOMA_VOZ        = 'pt-br'     # idioma para o gTTS
MODELO_WHISPER    = 'small'     # tiny | base | small | medium | large
MODELO_GEMINI     = 'gemini-2.0-flash'
DURACAO_GRAVACAO  = 120         # segundos de gravação (ajuste conforme a reunião)
NOME_REUNIAO      = 'Reunião de Alinhamento'  # aparece na ata

# Coloque sua chave do Gemini nos Secrets do Colab (ícone de chave na barra lateral)
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

print('✅ Configurações carregadas')

✅ Configurações carregadas


## 2. Gravação de áudio

In [ ]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
import datetime

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.onstop = async () => {
    blob = new Blob(chunks);
    text = await b2text(blob);
    resolve(text);
  };
  recorder.stop();
});
"""

def gravar_audio(segundos=60):
    print(f'🎙️ Gravando por {segundos} segundos... Fale agora!')
    display(Javascript(RECORD_JS))
    resultado = output.eval_js(f'record({segundos * 1000})')
    audio_bytes = b64decode(resultado.split(',')[1])
    caminho = '/content/reuniao_audio.wav'
    with open(caminho, 'wb') as f:
        f.write(audio_bytes)
    print('✅ Gravação concluída!')
    display(Audio(caminho, autoplay=False))
    return caminho

# Registra horário de início
horario_inicio = datetime.datetime.now().strftime('%d/%m/%Y às %H:%M')

arquivo_audio = gravar_audio(DURACAO_GRAVACAO)

🎙️ Gravando por 20 segundos... Fale agora!


<IPython.core.display.Javascript object>

✅ Gravação concluída!


## 3. Transcrição com Whisper

In [ ]:
import whisper

print(f'⏳ Carregando modelo Whisper ({MODELO_WHISPER})...')
modelo_whisper = whisper.load_model(MODELO_WHISPER)

print('🔎 Transcrevendo...')
resultado = modelo_whisper.transcribe(
    arquivo_audio,
    fp16=False,
    language=IDIOMA_GRAVACAO
)

transcricao = resultado['text']

print('\n--- TRANSCRIÇÃO ---')
print(transcricao)

⏳ Carregando modelo Whisper (small)...


100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 91.7MiB/s]


🔎 Transcrevendo...

--- TRANSCRIÇÃO ---
 Oi, hoje eu vou falar sobre a minha prova que eu vou ter ela amanhã e é sobre rede. E então estamos aprendendo sobre comunicação entre redes, arquiteturas de redes e muitas outras coisas que vão ser importantes para...


## 4. Análise inteligente com Gemini

In [ ]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.4 MB/s eta 0:00:00


In [ ]:
import json, re, time
from groq import Groq
from google.colab import userdata

# Busca a chave salva nos 'Secrets' (ícone de chave no menu lateral esquerdo)
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    cliente_groq = Groq(api_key=GROQ_API_KEY)
except Exception as e:
    print("❌ Erro: Certifique-se de que o nome da Secret é 'GROQ_API_KEY' e que o acesso está ativado.")
    raise e

# Mantemos exatamente o seu prompt original
PROMPT_ATA = f"""
Você é um assistente especializado em análise de reuniões corporativas.
Analise a transcrição abaixo e retorne um JSON com a seguinte estrutura:

{{
  "resumo": "Parágrafo com o resumo executivo da reunião",
  "participantes_mencionados": ["lista de nomes mencionados na transcrição"],
  "decisoes": [
    "Decisão 1 tomada na reunião",
    "Decisão 2 tomada na reunião"
  ],
  "tarefas": [
    {{"responsavel": "Nome", "tarefa": "descrição", "prazo": "prazo se mencionado ou 'A definir'"}}
  ],
  "proximos_passos": [
    "Próximo passo 1",
    "Próximo passo 2"
  ],
  "confirmacao_voz": "Frase curta e natural para ser lida em voz alta confirmando que a ata foi gerada. Ex: Ata gerada com sucesso! Foram identificadas 3 tarefas e 2 decisões."
}}

Responda APENAS com o JSON, sem markdown, sem explicações.

TRANSCRIÇÃO:
{transcricao}
"""

print('🚀 Analisando reunião com Groq (Llama 3)...')

try:
    # Chamada ao modelo Llama 3 no Groq
    resposta = cliente_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": PROMPT_ATA}
        ],
        temperature=0.1 # Baixa temperatura para o JSON vir mais "rígido"
    )

    texto_resposta = resposta.choices[0].message.content.strip()

    # Limpeza de markdown
    texto_resposta = re.sub(r'^```json\s*', '', texto_resposta)
    texto_resposta = re.sub(r'^```\s*', '', texto_resposta)
    texto_resposta = re.sub(r'\s*```$', '', texto_resposta)

    dados_ata = json.loads(texto_resposta)

    print('\n✅ Análise concluída!')
    print(f"📋 Resumo: {dados_ata['resumo'][:120]}...")
    print(f"✅ Decisões: {len(dados_ata['decisoes'])}")
    print(f"📌 Tarefas: {len(dados_ata['tarefas'])}")

except Exception as e:
    print(f"❌ Ocorreu um erro na análise: {e}")

🚀 Analisando reunião com Groq (Llama 3)...

✅ Análise concluída!
📋 Resumo: A reunião tratou sobre uma prova de rede que um dos participantes terá amanhã, abordando tópicos como comunicação entre ...
✅ Decisões: 0
📌 Tarefas: 1


## 5. Confirmação por voz (gTTS)

In [ ]:
import re as _re
from gtts import gTTS
from IPython.display import Audio, display

frase_confirmacao = dados_ata.get('confirmacao_voz', 'Ata de reunião gerada com sucesso!')
frase_limpa = _re.sub(r'\*', '', frase_confirmacao)

print(f'🔊 Reproduzindo: "{frase_limpa}"')

tts = gTTS(text=frase_limpa, lang=IDIOMA_VOZ, slow=False)
caminho_audio_resposta = '/content/confirmacao.mp3'
tts.save(caminho_audio_resposta)

display(Audio(caminho_audio_resposta, autoplay=True))

🔊 Reproduzindo: "Ata gerada com sucesso! Foram identificadas 0 decisões e 1 tarefa."


## 6. Geração da Ata em PDF

In [ ]:
from fpdf import FPDF
import datetime
import re

# Função para remover emojis e caracteres que o PDF padrão não entende
def limpar_texto(texto):
    if not texto: return ""
    # Remove emojis e caracteres especiais fora do range Latin-1
    return re.sub(r'[^\x00-\xff]', '', str(texto))

class AtaPDF(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 15)
        self.set_text_color(30, 30, 30)
        # Removi o emoji de microfone aqui
        self.cell(0, 10, 'Ata de Reuniao - Gerada por IA', align='C', new_x='LMARGIN', new_y='NEXT')
        self.set_draw_color(100, 100, 200)
        self.set_line_width(0.5)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.set_text_color(150, 150, 150)
        self.cell(0, 10, f'Pagina {self.page_no()} - Gerado automaticamente', align='C')

    def secao(self, titulo):
        self.ln(4)
        self.set_fill_color(240, 240, 255)
        self.set_font('Helvetica', 'B', 11)
        self.set_text_color(50, 50, 150)
        # Limpamos o título da seção (onde tinham os emojis)
        self.cell(0, 8, f'  {limpar_texto(titulo)}', fill=True, new_x='LMARGIN', new_y='NEXT')
        self.ln(2)
        self.set_text_color(30, 30, 30)

    def item(self, texto, bullet='-'):
        self.set_font('Helvetica', '', 10)
        self.multi_cell(0, 6, f'  {bullet} {limpar_texto(texto)}', new_x='LMARGIN', new_y='NEXT')

    def paragrafo(self, texto):
        self.set_font('Helvetica', '', 10)
        self.multi_cell(0, 6, limpar_texto(texto), new_x='LMARGIN', new_y='NEXT')


def gerar_pdf(dados, nome_reuniao, horario):
    pdf = AtaPDF()
    pdf.add_page()

    # Metadados
    pdf.set_font('Helvetica', '', 10)
    pdf.set_text_color(80, 80, 80)
    pdf.cell(0, 6, f'Reuniao: {limpar_texto(nome_reuniao)}', new_x='LMARGIN', new_y='NEXT')
    pdf.cell(0, 6, f'Data/Hora: {horario}', new_x='LMARGIN', new_y='NEXT')

    participantes = ', '.join(dados.get('participantes_mencionados', [])) or 'Nao identificados'
    pdf.cell(0, 6, f'Participantes: {limpar_texto(participantes)}', new_x='LMARGIN', new_y='NEXT')
    pdf.ln(2)

    # Resumo
    pdf.secao('Resumo Executivo')
    pdf.paragrafo(dados.get('resumo', ''))

    # Decisões
    pdf.secao('Decisoes Tomadas')
    decisoes = dados.get('decisoes', [])
    if decisoes:
        for d in decisoes:
            pdf.item(d)
    else:
        pdf.paragrafo('  Nenhuma decisao identificada.')

    # Tarefas
    pdf.secao('Tarefas e Responsaveis')
    tarefas = dados.get('tarefas', [])
    if tarefas:
        for t in tarefas:
            resp = t.get('responsavel', '?')
            task = t.get('tarefa', '')
            prazo = t.get('prazo', 'A definir')
            linha = f"{resp} - {task} (Prazo: {prazo})"
            pdf.item(linha, bullet='>')
    else:
        pdf.paragrafo('  Nenhuma tarefa identificada.')

    # Próximos passos
    pdf.secao('Proximos Passos')
    proximos = dados.get('proximos_passos', [])
    if proximos:
        for p in proximos:
            pdf.item(p)
    else:
        pdf.paragrafo('  Nenhum proximo passo identificado.')

    # Transcrição completa
    pdf.secao('Transcricao Completa')
    pdf.set_font('Helvetica', 'I', 9)
    pdf.set_text_color(80, 80, 80)
    pdf.multi_cell(0, 5, limpar_texto(transcricao), new_x='LMARGIN', new_y='NEXT')

    caminho_pdf = '/content/ata_reuniao.pdf'
    pdf.output(caminho_pdf)
    return caminho_pdf

# Chama a função
caminho_pdf = gerar_pdf(dados_ata, "Nome da Reuniao", horario_inicio)
print(f'✅ PDF gerado: {caminho_pdf}')

from google.colab import files
files.download(caminho_pdf)

✅ PDF gerado: /content/ata_reuniao.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. (Bônus) Visualizar o resumo no notebook

In [ ]:
from IPython.display import Markdown, display

linhas = []
linhas.append(f'# 📋 Ata — {NOME_REUNIAO}')
linhas.append(f'**Data:** {horario_inicio}')
linhas.append('')
linhas.append('## Resumo')
linhas.append(dados_ata.get('resumo', ''))
linhas.append('')
linhas.append('## ✅ Decisões')
for d in dados_ata.get('decisoes', []):
    linhas.append(f'- {d}')
linhas.append('')
linhas.append('## 📌 Tarefas')
for t in dados_ata.get('tarefas', []):
    linhas.append(f'- **{t["responsavel"]}** → {t["tarefa"]} *(Prazo: {t["prazo"]})*')
linhas.append('')
linhas.append('## 🚀 Próximos Passos')
for p in dados_ata.get('proximos_passos', []):
    linhas.append(f'- {p}')

display(Markdown('\n'.join(linhas)))

---
## 8. (Bônus) Loop contínuo — reunião longa em múltiplos blocos

Para reuniões longas, grave em blocos e vá acumulando a transcrição.

In [ ]:
# Execute esta célula várias vezes durante a reunião.
# Cada gravação é adicionada à transcrição acumulada.

BLOCO_SEGUNDOS = 60  # duração de cada bloco

if 'transcricao_acumulada' not in dir():
    transcricao_acumulada = ''
    bloco_num = 0

bloco_num += 1
print(f'--- Bloco {bloco_num} ---')

arquivo_bloco = gravar_audio(BLOCO_SEGUNDOS)

resultado_bloco = modelo_whisper.transcribe(arquivo_bloco, fp16=False, language=IDIOMA_GRAVACAO)
transcricao_acumulada += f'\n[Bloco {bloco_num}] ' + resultado_bloco['text']

print(f'✅ Bloco {bloco_num} adicionado. Total de caracteres: {len(transcricao_acumulada)}')
print('\nPara gerar a ata final, defina: transcricao = transcricao_acumulada')
print('e execute novamente as células 4, 5 e 6.')

--- Bloco 1 ---
🎙️ Gravando por 60 segundos... Fale agora!


<IPython.core.display.Javascript object>

KeyboardInterrupt: 